# Football Oracle — Experiments

Experiments on match odds and team ratings across five seasons of La Liga, Premier League, Bundesliga, Serie A, and Ligue 1.

**Contents**

1. Setup
2. Data preparation
3. Exploration
4. Discriminator analysis

# 1. Setup

In [ ]:
%pip install torch
%pip install -U scikit-learn

In [ ]:
import pandas as pd
import numpy as np
from statsbombpy import sb
from itertools import groupby
import matplotlib.pyplot as plt
import torch

# 2. Data preparation

In [ ]:
df = pd.read_csv("football_matches_odds.csv")
match_features = pd.read_csv("match_features.csv")

cols = [
    "home_gk_rating", "home_def_avg", "home_mid_avg", "home_att_avg",
    "away_gk_rating", "away_def_avg", "away_mid_avg", "away_att_avg",
    "diff_def", "diff_mid", "diff_att",
    "home_elo", "away_elo", "elo_diff",
]

df[cols] = match_features[cols]

conditions = [
    (df["odds_home_win"] <= df["odds_draw"]) & (df["odds_home_win"] <= df["odds_away_win"]),
    (df["odds_away_win"] <= df["odds_draw"]) & (df["odds_home_win"] >= df["odds_away_win"]),
    (df["odds_away_win"] >= df["odds_draw"]) & (df["odds_home_win"] >= df["odds_draw"])
]

values = ["home", "away", "draw"]

df["pred"] = np.select(conditions, values, default=None)

dt = pd.to_datetime(df["date"])
df["month"] = dt.dt.month
df["day"] = dt.dt.day

t = pd.to_datetime(df["time"], format="%H:%M")
df["time_mins"] = t.dt.hour * 60 + t.dt.minute


In [ ]:
conditions = [
    (df["league"] == "La Liga"),
    (df["league"] == "Premier League"),
    (df["league"] == "Bundesliga"),
    (df["league"] == "Serie A"),
    (df["league"] == "Ligue 1")
]
values = [1,2,3,4,5]

df["league_enc"] = np.select(conditions, values, default=None)

In [ ]:
inv_home = 1 / df["odds_home_win"]
inv_draw = 1 / df["odds_draw"]
inv_away = 1 / df["odds_away_win"]
total = inv_home + inv_draw + inv_away

df["prob_home"] = inv_home / total
df["prob_draw"] = inv_draw / total
df["prob_away"] = inv_away / total

df["guessed_correct"] = df["pred"] == df["winner"]

df["prob_taken"] = df[["prob_home", "prob_draw", "prob_away"]].max(axis=1)

# 3. Exploration

In [ ]:
df.iloc[0]

In [ ]:
df

In [ ]:
print(f"Total correct guesses (odds): {df["guessed_correct"].sum()} out of {len(df)}")
print(f"Total correct guesses (only home): {(df["winner"] == "home").sum()} out of {len(df)}")

leagues = {
    "SP1": "La Liga",
    "E0": "Premier League",
    "D1": "Bundesliga",
    "I1": "Serie A",
    "F1": "Ligue 1"
}

for code, name in leagues.items():
    league_df = df[df["league"] == name]
    accuracy = league_df["guessed_correct"].mean() * 100
    only_home_acc = (league_df["winner"] == "home").mean() * 100
    
    print(name)
    print(f"-accuracy (odds): {accuracy:.2f}%")
    print(f"-accuracy (only home): {only_home_acc:.2f}%")
    





In [ ]:
streaks = {
    value: max(
        (len(list(group)) for k, group in groupby(df["guessed_correct"]) if k == value),
        default=0
    )
    for value in [True, False]
}

print(f"Max consecutive correct: {streaks[True]}")
print(f"Max consecutive incorrect: {streaks[False]}")

In [ ]:
df[df["guessed_correct"] == 0]["odds_home_win"].plot.hist(bins=100, edgecolor="black")
plt.show()

print(df[df["guessed_correct"] == 0]["odds_home_win"].mean())
print(df[df["guessed_correct"] == 0]["odds_home_win"].std())

In [ ]:
df[df["guessed_correct"] == 0]["odds_away_win"].plot.hist(bins=100, edgecolor="black")
plt.show()
print(df[df["guessed_correct"] == 0]["odds_away_win"].mean())
print(df[df["guessed_correct"] == 0]["odds_away_win"].std())

In [ ]:
df[df["guessed_correct"] == 0]["odds_draw"].plot.hist(bins=100, edgecolor="black")
plt.show()
print(df[df["guessed_correct"] == 0]["odds_draw"].mean())
print(df[df["guessed_correct"] == 0]["odds_draw"].std())

# 4. Discriminator analysis

In [ ]:
from sklearn.metrics import accuracy_score

ds = df
accs = []
thrs = [0.45, 0.5, 0.55, 0.6, 0.65, 0.7, 0.75, 0.8]
for thr in thrs:
    thr_preds = np.select(
        [ds["pred"] == "home", ds["pred"] == "draw", ds["pred"] == "away"],
        [ds["prob_home"] >= thr, ds["prob_draw"] >= thr, ds["prob_away"] >= thr],
    )
    accs.append(accuracy_score(ds["guessed_correct"], thr_preds))

plt.plot(thrs, accs)
plt.show()

In [ ]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

data_split_ratio = 0.8

features = [ 
            "time_mins", 
            "prob_home", 
            "prob_draw", 
            "prob_away", 
            "market_avg_home_win", 
            "market_avg_away_win", 
            "market_avg_draw", 
            "month", 
            "day",
            "league_enc",
            "odds_home_win",
            "odds_away_win",
            "odds_draw"
           ]

split = int(data_split_ratio * len(df))
train_raw = df.iloc[:split].dropna().copy()
test_raw  = df.iloc[split:].dropna().copy()

train, test = train_raw.copy(), test_raw.copy()
scaler = StandardScaler()
train[features] = scaler.fit_transform(train[features])
test[features]  = scaler.transform(test[features])

y_train, y_test = train_raw["guessed_correct"], test_raw["guessed_correct"]

logreg = LogisticRegression(random_state=42, max_iter=1000).fit(train[features], y_train)
logreg_preds = logreg.predict(test[features])

tree = DecisionTreeClassifier(max_depth=4, min_samples_leaf=50, random_state=42)
tree.fit(train_raw[features], y_train)
tree_preds = tree.predict(test_raw[features])

thr = 0.5
thr_preds = np.select(
    [test_raw["pred"] == "home", test_raw["pred"] == "draw", test_raw["pred"] == "away"],
    [test_raw["prob_home"] >= thr, test_raw["prob_draw"] >= thr, test_raw["prob_away"] >= thr],
    default=False,
)

print(f"Threshold {thr} accuracy: {accuracy_score(y_test, thr_preds):.4f}")
print(f"Discriminator (logreg) accuracy: {accuracy_score(y_test, logreg_preds):.4f}")
print(f"Discriminator (tree) accuracy: {accuracy_score(y_test, tree_preds):.4f}")

In [ ]:
#Tree and logreg disagreemetns
disag = tree_preds != logreg_preds
agree = tree_preds == logreg_preds
print("Disagreements: " + str(disag.sum()))
test["logreg"] = logreg_preds
test["tree"] = tree_preds
print("Logreg was right: " + str((test[disag]["guessed_correct"] == test[disag]["logreg"]).sum()))
print("Tree was right: " + str((test[disag]["guessed_correct"] == test[disag]["tree"]).sum()))
print("Correct guesses duing disagreement: " + str(test[disag]["guessed_correct"].mean() * 100) + "%")
test[agree]["prob_taken"].mean()

In [ ]:
#Tree, logreg and treshold disagreements
disagree = (logreg_preds != tree_preds) | (tree_preds != thr_preds)
print("Disagreements: " + str(disagree.sum()))
test["logreg"] = logreg_preds
test["tree"] = tree_preds
test["thrsh"] = thr_preds
print("Logreg was right: " + str((test[disagree]["guessed_correct"] == test[disagree]["logreg"]).sum()))
print("Tree was right: " + str((test[disagree]["guessed_correct"] == test[disagree]["tree"]).sum()))
print("Threshold was right: " + str((test[disagree]["guessed_correct"] == test[disagree]["thrsh"]).sum()))
print("Correct guesses duing disagreement: " + str(test[disagree]["guessed_correct"].mean() * 100) + "%")

In [ ]:
evaluation_game = {
    "time_mins": 14 * 60,
    "prob_home": (1/1.11) / ((1/1.11) + (1/10.5) + (1/8)),
    "prob_draw": (1/8) / ((1/1.11) + (1/10.5) + (1/8)),
    "prob_away": (1/10.5) / ((1/1.11) + (1/10.5) + (1/8)),
    "market_avg_home_win": 1.11, 
    "market_avg_away_win": 10.5, 
    "market_avg_draw": 8, 
    "month": 11,
    "day": 22,
    "league_enc": 1,
    "odds_home_win": 1.11,
    "odds_away_win": 10.5,
    "odds_draw": 8
    
}
print(evaluation_game["prob_home"])
evaluation_game_df = pd.DataFrame([evaluation_game])
print(logreg.predict(evaluation_game_df))
print(tree.predict(evaluation_game_df))

In [ ]:
print(logreg_preds.sum())
print((~logreg_preds).sum())
print(tree_preds.sum())
print((~tree_preds).sum())
print(thr_preds.sum())
print((~thr_preds).sum())

## Odds distribution - no discriminator vs discriminator

In [ ]:
df["odds_taken"] = np.select(
    [df["pred"] == "home", df["pred"] == "draw", df["pred"] == "away"],
    [df["odds_home_win"], df["odds_draw"], df["odds_away_win"]],
    default=np.nan,
)

test_pool = pd.DataFrame({
    "odds_taken": np.select(
        [test_raw["pred"] == "home", test_raw["pred"] == "draw", test_raw["pred"] == "away"],
        [test_raw["odds_home_win"], test_raw["odds_draw"], test_raw["odds_away_win"]],
        default=np.nan,
    ),
    "won": test_raw["guessed_correct"],
})

groups = {
    "All games (no discriminator)": pd.DataFrame({
        "odds_taken": df["odds_taken"],
        "won": df["guessed_correct"],
    }),
    "Threshold (prob ≥ 0.50)": test_pool[thr_preds],
    "Logistic regression": test_pool[logreg_preds],
    "Decision tree": test_pool[tree_preds],
}

summary = pd.DataFrame([
    {
        "Selection": name,
        "Games": len(group),
        "Selected %": len(group)
        / (len(df) if name.startswith("All games") else len(test_pool))
        * 100,
        "Mean odds": group["odds_taken"].mean(),
        "Median odds": group["odds_taken"].median(),
        "P10 odds": group["odds_taken"].quantile(0.10),
        "P90 odds": group["odds_taken"].quantile(0.90),
        "Hit rate %": group["won"].mean() * 100,
    }
    for name, group in groups.items()
]).set_index("Selection")

display(summary.round(2))

print(
    f"All test-window games (fair baseline for the discriminators): "
    f"n={len(test_pool):,}, mean odds={test_pool['odds_taken'].mean():.2f}, "
    f"median={test_pool['odds_taken'].median():.2f}, "
    f"hit rate={test_pool['won'].mean():.1%}"
)

In [ ]:
SURFACE = "#fcfcfb"
INK = "#0b0b0b"
INK_2 = "#52514e"
MUTED = "#898781"
GRID = "#e1e0d9"
AXIS = "#c3c2b7"

COLORS = {
    "All games (no discriminator)": MUTED,
    "Threshold (prob ≥ 0.50)": "#2a78d6",
    "Logistic regression": "#eb6834",
    "Decision tree": "#1baf7a",
}

bin_edges = np.arange(1.0, 2.85 + 0.05, 0.10)

fig, (ax_hist, ax_ecdf) = plt.subplots(1, 2, figsize=(13, 5.2), layout="constrained")
fig.patch.set_facecolor(SURFACE)

for ax in (ax_hist, ax_ecdf):
    ax.set_facecolor(SURFACE)
    ax.grid(axis="y", color=GRID, linewidth=0.8)
    ax.set_axisbelow(True)
    ax.tick_params(colors=INK_2, labelsize=9, length=0)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    for side in ("left", "bottom"):
        ax.spines[side].set_color(AXIS)
    ax.set_xlim(1.0, 2.85)
    ax.set_xlabel("Decimal odds of the backed outcome", color=INK_2, fontsize=10)

for name, group in groups.items():
    odds = group["odds_taken"]
    ax_hist.hist(
        odds,
        bins=bin_edges,
        weights=np.full(len(odds), 100 / len(odds)),
        histtype="step",
        color=COLORS[name],
        linewidth=2,
        label=f"{name} (n={len(odds):,})",
    )
ax_hist.set_ylim(bottom=0)
ax_hist.set_ylabel("% of selection per 0.10 odds", color=INK_2, fontsize=10)
ax_hist.set_title(
    "Distribution of backed odds", loc="left", fontsize=12,
    fontweight="bold", color=INK,
)

for name, group in groups.items():
    odds = np.sort(group["odds_taken"].to_numpy())
    cumulative = np.arange(1, len(odds) + 1) / len(odds) * 100
    ax_ecdf.step(odds, cumulative, where="post", color=COLORS[name], linewidth=2)
ax_ecdf.axhline(50, color=AXIS, linewidth=0.8, linestyle="--")
ax_ecdf.set_ylim(0, 100)
ax_ecdf.set_ylabel("Cumulative % of selection", color=INK_2, fontsize=10)
ax_ecdf.set_title(
    "Cumulative distribution", loc="left", fontsize=12,
    fontweight="bold", color=INK,
)

handles, labels = ax_hist.get_legend_handles_labels()
fig.legend(
    handles, labels, loc="outside lower center", ncol=4,
    frameon=False, fontsize=9, labelcolor=INK_2,
)
fig.suptitle(
    "Discriminator selection shifts bets toward shorter odds",
    x=0.01, ha="left", fontsize=14, fontweight="bold", color=INK,
)
plt.show()